# DataEngine Backtest Notebook

This notebook runs a backtest using the same engine logic implemented in `data_engine.py` (mapping, timestamp parsing, expiry handling, and `run_simulation_on_signal`).

Default source is Google Sheets using the same hardcoded sheet URLs as `app.py`, with CSV fallback available if needed.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

repo_root = Path.cwd()
if not (repo_root / 'data_engine.py').exists() and (repo_root.parent / 'data_engine.py').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from data_engine import DataEngine

/Users/azturan.b/algo_trade/tracker/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Load signals from local CSV only
CSV_PATH = Path('../data/TradeSignals - Stocks.csv')
if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

df_to_process = pd.read_csv(CSV_PATH, dtype=str)
is_gsheet_source = False
print("Loaded CSV rows:", len(df_to_process))

Loaded CSV rows: 1760


In [3]:
engine = DataEngine()

if is_gsheet_source:
    headers = list(df_to_process.columns)
    mapping = engine.auto_map_columns(headers)
    print('Header count:', len(headers))
    print('Mapped fields:', len(mapping))
else:
    headers, encoding, sep = engine.load_csv_headers(CSV_PATH)
    mapping = engine.auto_map_columns(headers)
    print('Detected encoding:', encoding)
    print('Detected separator:', repr(sep))
    print('Header count:', len(headers))
    print('Mapped fields:', len(mapping))

mapping

Detected encoding: ascii
Detected separator: ','
Header count: 20
Mapped fields: 10


{'Signal Timestamp': 'Date',
 'Signal Timeframe': 'Timeframe',
 'Symbol': 'Symbol',
 'Direction': 'Direction',
 'Signal Price': 'Signal Price',
 'Optimum Price': 'Optimum Price',
 'Stop Loss Price': 'Stop Loss Price',
 'Take Profit Price': 'TP Price',
 'Break Possibility': 'Breakout Possibility',
 'Current Price': 'Current Price'}

In [4]:
if is_gsheet_source:
    ok, msg = engine.process_dataframe(df_to_process, mapping, is_gsheet=True)
else:
    ok, msg = engine.load_csv_and_process(CSV_PATH, mapping, encoding, sep)

print(msg)

if not ok:
    raise RuntimeError(msg)

signals = engine.get_signals()
signals_df = pd.DataFrame(signals)
print('Signals loaded:', len(signals_df))
signals_df.head(5)

Processed 1760 valid signals.
Signals loaded: 1760


,Signal Timestamp,Signal Timeframe,Symbol,Direction,Signal Price,Optimum Price,Stop Loss Price,Take Profit Price,Break Possibility,Current Price,Original Index
0,2025-10-24 07:47:20+00:00,Daily,VRSN,Long,237.00000,225.04044,213.13863,269.70609,21.0,308.00,0
1,2025-10-24 08:26:39+00:00,4-hour,KLAC,Short,1186.98600,1297.01466,1352.35445,1083.60870,50.0,1977.22,1
2,2025-10-24 08:49:29+00:00,1-hour,UNP,Long,217.78313,209.88837,206.61176,224.82000,15.0,271.24,2
3,2025-10-24 09:08:14+00:00,1-hour,ANET,Short,155.21240,168.07713,171.92615,148.48435,10.0,159.34,3
4,2025-10-24 10:10:04+00:00,1-hour,MU,Short,217.31325,240.19250,246.42759,203.63043,40.0,877.87,4


In [5]:
# Backtest parameters (aligned with DataEngine.run_simulation_on_signal).
EXPIRY_BARS = 20
SIM_BARS = 30
TF_OVERRIDE = None  # Example: '1d', '1h', '15m'
PRICE_MODE = 'close'  # close|open|high|low|optimistic|pessimistic
MAX_SIGNALS = 200

# Date filter: only use signals on/after this date (YYYY-MM-DD)
START_DATE = '2026-01-01'

# Enable Monte Carlo simulation
MONTE_CARLO_OPTS = {
    'enabled': True,
    'reshuffle': True,
    'noise_pct': 1.0,
}
MC_ITERATIONS = 20

print('Parameters ready. START_DATE =', START_DATE)

Parameters ready. START_DATE = 2026-01-01


In [6]:
def extract_final_pnl(series):
    if not series:
        return np.nan
    return float(series[-1])

rows = []
# Use most recent signals first, but filter by START_DATE if set
import pytz
start_dt = pd.to_datetime(START_DATE)
if start_dt.tzinfo is None:
    start_dt = start_dt.tz_localize(pytz.utc)

signals_filtered = [s for s in signals if (s.get('Signal Timestamp') is not None and s.get('Signal Timestamp') >= start_dt)]
print(f'Signals after {start_dt}:', len(signals_filtered))

signals_sorted = sorted(signals_filtered, key=lambda s: s.get('Signal Timestamp') or pd.NaT, reverse=True)
selected_signals = signals_sorted[:MAX_SIGNALS]

for i, sig in enumerate(selected_signals, start=1):
    result = engine.run_simulation_on_signal(
        signal=sig,
        expiry_bars=EXPIRY_BARS,
        sim_bars=SIM_BARS,
        tf_override=TF_OVERRIDE,
        price_mode=PRICE_MODE,
        monte_carlo_opts=MONTE_CARLO_OPTS,
        mc_iterations=MC_ITERATIONS,
    )

    if result is None:
        continue

    signal_res = result.get('Signal')
    optimum_res = result.get('Optimum')
    sl_res = result.get('SL')

    rows.append({
        'idx': i,
        'symbol': sig.get('Symbol'),
        'direction': sig.get('Direction'),
        'timeframe': sig.get('Signal Timeframe'),
        'timestamp': sig.get('Signal Timestamp'),
        'signal_final_pnl_pct': extract_final_pnl(signal_res.get('pnl') if signal_res else None),
        'signal_mae_pct': signal_res.get('mae') if signal_res else np.nan,
        'signal_mfe_pct': signal_res.get('mfe') if signal_res else np.nan,
        'optimum_final_pnl_pct': extract_final_pnl(optimum_res.get('pnl') if optimum_res else None),
        'sl_final_pnl_pct': extract_final_pnl(sl_res.get('pnl') if sl_res else None),
    })

results_df = pd.DataFrame(rows)
print(f'Computed simulations: {len(results_df)} / {len(selected_signals)}')
results_df.head(10)

Signals after 2026-01-01 00:00:00+00:00: 1159


Computed simulations: 200 / 200


,idx,symbol,direction,timeframe,timestamp,signal_final_pnl_pct,signal_mae_pct,signal_mfe_pct,optimum_final_pnl_pct,sl_final_pnl_pct
0,1,SBUX,Long,1-hour,2026-05-22 11:05:38+00:00,-1.529543,-3.935537,3.231686,NaN,NaN
1,2,FTNT,Short,4-hour,2026-05-22 11:02:03+00:00,2.679049,-0.897958,8.598201,NaN,NaN
2,3,LLY,Short,1-hour,2026-05-22 11:02:03+00:00,-5.912639,-9.069339,2.274683,NaN,NaN
3,4,SWKS,Short,1-hour,2026-05-22 10:09:08+00:00,-2.763813,-6.112384,6.294100,3.331303,NaN
4,5,SNPS,Short,1-hour,2026-05-22 10:02:03+00:00,9.742124,-0.118416,13.776315,NaN,NaN
5,6,ROST,Short,1-hour,2026-05-22 10:02:03+00:00,1.208647,-2.016061,7.155644,NaN,NaN
6,7,CRWD,Short,1-hour,2026-05-22 10:02:03+00:00,0.347366,-2.977702,7.223681,NaN,NaN
7,8,AMD,Short,1-hour,2026-05-22 09:13:33+00:00,-9.192551,-13.385117,5.566413,NaN,NaN
8,9,ANET,Short,1-hour,2026-05-22 09:02:03+00:00,-1.532682,-4.124907,6.891350,NaN,NaN
9,10,AAPL,Short,1-hour,2026-05-22 09:02:03+00:00,-0.322698,-2.672937,2.931715,NaN,NaN


In [7]:
if results_df.empty:
    raise RuntimeError('No simulation outputs. Check symbols/data availability.')

summary = results_df.agg({
    'signal_final_pnl_pct': ['count', 'mean', 'median', 'std', 'min', 'max'],
    'signal_mae_pct': ['mean', 'median', 'min', 'max'],
    'signal_mfe_pct': ['mean', 'median', 'min', 'max'],
})

print('Overall summary:')
display(summary)

by_tf = results_df.groupby('timeframe', dropna=False)['signal_final_pnl_pct'].agg(['count', 'mean', 'median'])
print('By timeframe:')
display(by_tf.sort_values('mean', ascending=False))

by_dir = results_df.groupby('direction', dropna=False)['signal_final_pnl_pct'].agg(['count', 'mean', 'median'])
print('By direction:')
display(by_dir.sort_values('mean', ascending=False))

Overall summary:


,signal_final_pnl_pct,signal_mae_pct,signal_mfe_pct
count,200.000000,NaN,NaN
mean,1.193878,-4.070616,8.151947
median,1.480566,-2.363551,7.431653
std,6.626891,NaN,NaN
min,-24.733478,-27.773604,0.641636
max,28.003769,0.000000,37.407058


By timeframe:


,count,mean,median
timeframe,,,
1-hour,146,1.661940,1.547013
4-hour,34,1.329339,0.748607
Daily,20,-2.453256,1.449586


By direction:


,count,mean,median
direction,,,
Long,82,4.604213,4.365302
Short,118,-1.176016,0.334932


In [8]:
# Optional: save results for later analysis.
OUTPUT_PATH = Path('backtest_results_from_data_engine.csv')
results_df.to_csv(OUTPUT_PATH, index=False)
print('Saved:', OUTPUT_PATH.resolve())

Saved: /Users/azturan.b/algo_trade/mean_reversion_repo/notebooks/backtest_results_from_data_engine.csv


In [9]:
import plotly.express as px
from IPython.display import HTML, display

# Histogram of final PnL
fig_hist = px.histogram(results_df, x='signal_final_pnl_pct', nbins=50, title='Distribution of Final PnL (%)')

# Boxplot by timeframe
fig_tf = px.box(results_df, x='timeframe', y='signal_final_pnl_pct', title='Final PnL by Timeframe')

# Boxplot by direction
fig_dir = px.box(results_df, x='direction', y='signal_final_pnl_pct', title='Final PnL by Direction')

# Display inline in the notebook
display(HTML(fig_hist.to_html(full_html=False, include_plotlyjs='cdn')))
display(HTML(fig_tf.to_html(full_html=False, include_plotlyjs=False)))
display(HTML(fig_dir.to_html(full_html=False, include_plotlyjs=False)))

# Save hist and charts as HTML fragments
OUT_HTML = 'backtest_plots.html'
with open(OUT_HTML, 'w') as f:
    f.write('<html><head><meta charset="utf-8"></head><body>')
    f.write(fig_hist.to_html(full_html=False, include_plotlyjs='cdn'))
    f.write(fig_tf.to_html(full_html=False, include_plotlyjs=False))
    f.write(fig_dir.to_html(full_html=False, include_plotlyjs=False))
    f.write('</body></html>')

print('Saved plots to', OUT_HTML)

# --- Portfolio simulation ---
# Parameters
START_BALANCE = 100000.0
BET_MODE = 'fraction'  # 'fraction' or 'fixed'
BET_SIZE = 0.01        # fraction of current balance when BET_MODE=='fraction'
FIXED_BET = 1000.0     # fixed USD bet when BET_MODE=='fixed'

# Prepare signals (ensure chronological order)
pf = results_df.copy()
if 'timestamp' in pf.columns:
    pf['timestamp'] = pd.to_datetime(pf['timestamp'])
    pf.sort_values('timestamp', inplace=True)
else:
    pf = pf.reset_index(drop=True)

balance = START_BALANCE
balances = []
dates = []
trade_pnls = []

for _, row in pf.iterrows():
    ret_pct = row.get('signal_final_pnl_pct', 0.0)
    if pd.isna(ret_pct):
        ret_pct = 0.0
    ret = float(ret_pct) / 100.0

    if BET_MODE == 'fraction':
        wager = max(1.0, balance * BET_SIZE)
    else:
        wager = FIXED_BET

    pnl = wager * ret
    balance += pnl

    balances.append(balance)
    dates.append(row.get('timestamp') if 'timestamp' in row.index else None)
    trade_pnls.append(pnl)

portfolio_df = pd.DataFrame({
    'timestamp': dates,
    'balance': balances,
    'trade_pnl': trade_pnls
})

# Metrics
final_balance = portfolio_df['balance'].iloc[-1] if not portfolio_df.empty else START_BALANCE
returns = portfolio_df['balance'].pct_change().fillna(0)
win_rate = (portfolio_df['trade_pnl'] > 0).sum() / max(1, len(portfolio_df))

# Max drawdown
cum = portfolio_df['balance']
peak = cum.cummax()
drawdown = (cum - peak) / peak
max_drawdown = drawdown.min() if not drawdown.empty else 0.0

print(f'Portfolio final balance: ${final_balance:,.2f}')
print(f'Win rate: {win_rate*100:.2f}%')
print(f'Max drawdown: {max_drawdown*100:.2f}%')

# Save portfolio results
PORT_OUT = 'backtest_portfolio.csv'
portfolio_df.to_csv(PORT_OUT, index=False)
print('Saved portfolio CSV to', PORT_OUT)

# Plot equity curve
fig_eq = px.line(portfolio_df, x='timestamp', y='balance', title='Portfolio Equity Curve')

# Display inline and append to HTML
display(HTML(fig_eq.to_html(full_html=False, include_plotlyjs=False)))
with open(OUT_HTML, 'a') as f:
    f.write(fig_eq.to_html(full_html=False, include_plotlyjs=False))

print('Appended equity plot to', OUT_HTML)

Saved plots to backtest_plots.html
Portfolio final balance: $102,411.87
Win rate: 67.50%
Max drawdown: -0.72%
Saved portfolio CSV to backtest_portfolio.csv


Appended equity plot to backtest_plots.html


In [10]:
# Check earliest and latest timestamps for signals, results, and portfolio
import pandas as pd

print('Kernel variables present:', [k for k in globals().keys() if k in ('signals_df','results_df','portfolio_df')])

if 'signals_df' in globals():
    s = pd.to_datetime(signals_df['Signal Timestamp'])
    print('\nSignals: count', len(signals_df))
    print('Earliest signal:', s.min())
    print('Latest signal :', s.max())
    display(signals_df.sort_values('Signal Timestamp')[['Signal Timestamp','Symbol']].head(5))
    display(signals_df.sort_values('Signal Timestamp')[['Signal Timestamp','Symbol']].tail(5))
else:
    print('signals_df not found')

if 'results_df' in globals():
    if 'timestamp' in results_df.columns:
        r = pd.to_datetime(results_df['timestamp'])
        print('\nResults: count', len(results_df))
        print('Earliest result timestamp:', r.min())
        print('Latest result timestamp :', r.max())
    else:
        print('\nresults_df exists but has no `timestamp` column')

if 'portfolio_df' in globals():
    p = pd.to_datetime(portfolio_df['timestamp'])
    print('\nPortfolio: count', len(portfolio_df))
    print('Earliest portfolio timestamp:', p.min())
    print('Latest portfolio timestamp :', p.max())
else:
    print('\nportfolio_df not found')

Kernel variables present: ['signals_df', 'results_df', 'portfolio_df']

Signals: count 1760
Earliest signal: 2025-09-26 07:36:11+00:00
Latest signal : 2026-05-22 11:05:38+00:00


,Signal Timestamp,Symbol
174,2025-09-26 07:36:11+00:00,SWKS
173,2025-09-26 07:36:11+00:00,TSLA
172,2025-09-26 07:36:11+00:00,AAPL
175,2025-09-26 08:11:21+00:00,IDXX
176,2025-09-26 11:05:11+00:00,PG


,Signal Timestamp,Symbol
1753,2026-05-22 10:02:03+00:00,SNPS
1756,2026-05-22 10:09:08+00:00,SWKS
1758,2026-05-22 11:02:03+00:00,LLY
1757,2026-05-22 11:02:03+00:00,FTNT
1759,2026-05-22 11:05:38+00:00,SBUX



Results: count 200
Earliest result timestamp: 2026-04-29 08:21:47+00:00
Latest result timestamp : 2026-05-22 11:05:38+00:00

Portfolio: count 200
Earliest portfolio timestamp: 2026-04-29 08:21:47+00:00
Latest portfolio timestamp : 2026-05-22 11:05:38+00:00
